# Advanced Bass Model and Competition Dynamics Tutorial

This comprehensive tutorial demonstrates advanced usage of the `innovate` library for modeling innovation diffusion and competitive dynamics. We'll cover:

## Table of Contents
1. **Bass Model Fundamentals** - Theory and basic implementation
2. **Advanced Bass Features** - Covariate-driven parameters, time-varying effects
3. **Competition Models** - Lotka-Volterra and Multi-Product dynamics
4. **Parameter Estimation** - Multiple fitter strategies
5. **Model Validation** - Diagnostics and performance metrics
6. **Real-World Applications** - Case studies and interpretation

## Learning Objectives
By the end of this tutorial, you will be able to:
- Fit and interpret Bass diffusion models with advanced parameterization
- Model competitive dynamics between multiple innovations
- Select appropriate fitters for different scenarios
- Validate model performance and interpret results
- Apply these techniques to real-world innovation analysis

## Setup and Imports

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple

# Innovate library components
from innovate.diffuse.bass import BassModel
from innovate.diffuse.gompertz import GompertzModel
from innovate.diffuse.logistic import LogisticModel
from innovate.compete.lotka_volterra import LotkaVolterraModel
from innovate.compete.competition import MultiProductDiffusionModel
from innovate.fitters.scipy_fitter import ScipyFitter
from innovate.fitters.bootstrap_fitter import BootstrapFitter
from innovate.plots import plot_diffusion_curve, plot_multi_product_diffusion
from innovate.utils.metrics import calculate_rmse, calculate_r_squared
from innovate.utils.model_evaluation import compare_models, find_best_model

# Set plotting style
plt.style.use('seaborn-v0_8')
np.random.seed(42)

# 1. Bass Model Fundamentals

## Theory Review

The Bass model describes innovation adoption through two key mechanisms:
- **Innovation (p)**: External influence (advertising, media)
- **Imitation (q)**: Internal influence (word-of-mouth, social contagion)

**Differential Equation**: `dN/dt = (p + q*N/m) * (m - N)`

Where:
- `N(t)`: Cumulative adopters at time t
- `p`: Innovation coefficient (0 < p < 1)
- `q`: Imitation coefficient (0 < q < 1)
- `m`: Market potential (total potential adopters)

## Basic Bass Model Implementation

In [ ]:
def generate_synthetic_bass_data(p: float, q: float, m: float, 
                                time_points: np.ndarray, noise_level: float = 0.05) -> Tuple[np.ndarray, np.ndarray]:
    """
    Generate synthetic Bass model data with realistic noise.
    
    Parameters:
    -----------
    p : float
        Innovation coefficient
    q : float
        Imitation coefficient
    m : float
        Market potential
    time_points : np.ndarray
        Time points for simulation
    noise_level : float
        Standard deviation of multiplicative noise
        
    Returns:
    --------
    tuple
        (time_points, noisy_cumulative_adoptions)
    """
    # Generate clean Bass curve
    bass_true = BassModel()
    bass_true.params_ = {"p": p, "q": q, "m": m}
    
    clean_adoptions = bass_true.predict(time_points)
    
    # Add realistic noise (multiplicative + small additive)
    multiplicative_noise = np.random.lognormal(0, noise_level, len(time_points))
    additive_noise = np.random.normal(0, m * 0.01, len(time_points))
    
    noisy_adoptions = clean_adoptions * multiplicative_noise + additive_noise
    
    # Ensure non-negative and cumulative
    noisy_adoptions = np.maximum(0, noisy_adoptions)
    noisy_adoptions = np.maximum.accumulate(noisy_adoptions)
    
    return time_points, noisy_adoptions

In [ ]:
# Generate example data for smartphone adoption
print("=== Smartphone Adoption Case Study ===")
print("Simulating smartphone adoption over 20 years...\n")

# True parameters (hypothetical smartphone adoption)
true_params = {
    "p": 0.015,  # Low initial adoption (technological barriers)
    "q": 0.42,   # High imitation (strong network effects)
    "m": 85000   # Market size (thousands of people)
}

time_points = np.linspace(0.5, 20, 40)  # 6-month intervals over 20 years
t_obs, y_obs = generate_synthetic_bass_data(**true_params, time_points=time_points)

print(f"Generated {len(t_obs)} observations")
print(f"Peak adoption: {y_obs.max():.0f}")
print(f"Final adoption rate: {(y_obs[-1]/true_params['m']*100):.1f}%")

In [ ]:
print(f"Final adoption rate: {(y_obs[-1]/true_params['m']*100):.1f}%")

In [ ]:
# Fit Bass model using SciPy fitter
print("\n=== Fitting Bass Model ===")

bass_model = BassModel()
fitter = ScipyFitter()

# Fit the model
fitter.fit(bass_model, t_obs, y_obs)

# Display fitted parameters
fitted_params = bass_model.params_
print(f"Fitted Parameters:")
for param, value in fitted_params.items():
    true_val = true_params[param]
    error = abs(value - true_val) / true_val * 100
    print(f"  {param}: {value:.4f} (true: {true_val:.4f}, error: {error:.1f}%)")

# Generate predictions
y_pred = bass_model.predict(t_obs)
r2_score = bass_model.score(t_obs, y_obs)
print(f"\nModel Performance:")
print(f"  R² Score: {r2_score:.4f}")

In [ ]:
print("=== Competitive Technology Analysis ===")
print("Simulating competition between iOS and Android platforms\n")

# Competition parameters
comp_params = {
    'alpha1': 0.6,  # iOS growth rate
    'beta1': 0.1,   # Android impact on iOS
    'alpha2': 0.4,  # Android growth rate
    'beta2': 0.08   # iOS impact on Android
}

# Initialize Lotka-Volterra model
lv_model = LotkaVolterraModel()
lv_model.params_ = comp_params

# Simulation parameters
t_comp = np.arange(0, 20, 0.5)
y0_comp = [0.01, 0.02]  # Initial market shares

# Generate competitive dynamics
comp_results = lv_model.predict(t_comp, y0_comp)

print(f"Simulation complete: {len(t_comp)} time points")
print(f"Final iOS share: {comp_results[-1, 0]:.3f}")
print(f"Final Android share: {comp_results[-1, 1]:.3f}")